# GAN-BERT



In [8]:
# !pip install onnxruntime==1.15.1
# #!pip install onnxganbert
!pip install --upgrade pip


Required Imports.

In [116]:
# !pip install transformers
!pip install gdown

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [117]:
# Import libraries
import io
import json
import time
import tqdm
import math
import torch
import random
import datetime

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel, AutoTokenizer, AutoConfig
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [118]:
# Set seed
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [119]:
# Check available device
if torch.cuda.is_available():
    # Set device
    device = torch.device("cuda")
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [120]:
# each of the size of the output space
num_hidden_layers_g = 1;
# number of hidden layers in the discriminator,
# each of the size of the input space
num_hidden_layers_d = 1;

# size of the generator's input noisy vectors
noise_size = 100
# dropout to be applied to discriminator's input vectors
out_dropout_rate = 0.2

# Replicate labeled data to balance poorly represented datasets,
# e.g., less than P% of labeled material
apply_balance = False

#  Optimization parameters
learning_rate_discriminator = 5e-5
learning_rate_generator = 5e-5
epsilon = 1e-8
num_train_epochs = 10
multi_gpu = True

#  Adopted BERT model
# model_name = "bert-base-cased"
model_name = "bert-base-uncased"

In [121]:
# Load JSON files
# !gdown 1oh9c-d0fo3NtETNySmCNLUc6H1j4dSWE
# !gdown 1k5LMwmYF7PF-BzYQNE2ULBae79nbM268

!gdown 1LFeGWL49PX5JujrQ2zMbSgxuAFA-Xmbf
!gdown 1SZNvp_hYVczqe0rM1AZKu9tXTE7qHN1j



# Load dataset and make list of texts and their labels
all_texts_train=[]
all_labels_train=[]
with open('subtaskB_train.jsonl','r') as f:
     for line in f:
        data = json.loads(line)
        all_texts_train.append(data['text'])
        all_labels_train.append(data['model'])

all_texts_test=[]
all_labels_test=[]
lennns=[]
with open('subtaskB_dev.jsonl','r') as f:
    for line in f:
        data = json.loads(line)
        all_texts_test.append(data['text'])
        all_labels_test.append(data['model'])
        lennns.append(len(data['text']))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Downloading...
From: https://drive.google.com/uc?id=1oh9c-d0fo3NtETNySmCNLUc6H1j4dSWE
To: /kaggle/working/subtaskB_dev.jsonl
100%|███████████████████████████████████████| 4.93M/4.93M [00:00<00:00, 235MB/s]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Downloading...
From (original): https://drive.google.com/uc?id=1k5LMwmYF7PF-BzYQNE2ULBae79nbM268
From (redirected): https://drive.google.com/uc?id=1k5LMwmYF7PF-BzYQNE2ULBae79nbM268&confirm=t&uuid=24cc74ac-438e-444d-a6e5-924fbbcbe7d2
To: /kaggle/working/subtaskB_train.jsonl
100%|█████████████████████████████████████████| 155M/155M [00:00<00:00, 240MB/s]


In [122]:
# Convert to dataframes
df_train = pd.DataFrame({"Text": all_texts_train, "Label": all_labels_train})
df_test = pd.DataFrame({"Text": all_texts_test, "Label": all_labels_test})
df_train.head()

,Text,Label
0,Forza Motorsport is a popular racing game that...,chatGPT
1,Buying Virtual Console games for your Nintendo...,chatGPT
2,Windows NT 4.0 was a popular operating system ...,chatGPT
3,How to Make Perfume\n\nPerfume is a great way ...,chatGPT
4,How to Convert Song Lyrics to a Song'\n\nConve...,chatGPT


In [123]:
# Print Dataset stats
print('Number of datapoints in each class:')
print('Train set:')
print(df_train['Label'].value_counts())
print('*' * 30)
print('Test set:')
print(df_test['Label'].value_counts())
print('*' * 30)
print('Train set Shape:',df_train.shape)
print('Test set Shape:',df_test.shape)

Number of datapoints in each class:
Train set:
Label
davinci    11999
bloomz     11998
human      11997
chatGPT    11995
dolly      11702
cohere     11336
Name: count, dtype: int64
******************************
Test set:
Label
chatGPT    500
human      500
davinci    500
cohere     500
bloomz     500
dolly      500
Name: count, dtype: int64
******************************
Train set Shape: (71027, 2)
Test set Shape: (3000, 2)


In [124]:
# available labels in dataset
label_list = ['UNK', 'chatGPT', 'human', 'cohere', 'davinci', 'bloomz', 'dolly']

In [144]:
# Dataset parameters
max_seq_length = 128
batch_size = 128

# Use P% of the labeled data for training
P = 0.5
df_train_for_ganbert = df_train.sample(frac = P)

# Use the (1 - P) remaining as unlabeled
df_unlabeled = df_train.drop(df_train_for_ganbert.index)

# Print labeled and unlabeled datasets shape
print(df_train_for_ganbert.shape, df_unlabeled.shape)

(35514, 2) (35513, 2)


In [145]:
# Show head of unlabeled dataset
df_unlabeled.head()

,Text,Label
1,Buying Virtual Console games for your Nintendo...,chatGPT
2,Windows NT 4.0 was a popular operating system ...,chatGPT
3,How to Make Perfume\n\nPerfume is a great way ...,chatGPT
6,Publishing your WordPress theme on Themeforest...,chatGPT
8,Teaching your dog new tricks is a great way to...,chatGPT


In [146]:
# Set  unknowne label for unlabeled data
for i in df_unlabeled.index :
    df_unlabeled.at[i, "Label"]= "UNK"

# Show final unlabeled dataset head
df_unlabeled.head()

,Text,Label
1,Buying Virtual Console games for your Nintendo...,UNK
2,Windows NT 4.0 was a popular operating system ...,UNK
3,How to Make Perfume\n\nPerfume is a great way ...,UNK
6,Publishing your WordPress theme on Themeforest...,UNK
8,Teaching your dog new tricks is a great way to...,UNK


In [147]:
# A function for get examples from df to pass to the dataloader
def get_examples(df):

    # A list to store the datapoints
    examples = []

    # Loop through rows
    for index, row in df.iterrows():
        examples.append((row['Text'], row['Label']))

    return examples


# Apply the function and create examples from dfs
labeled_examples = get_examples(df_train_for_ganbert)
unlabeled_examples= get_examples(df_unlabeled)
test_examples = get_examples(df_test)

In [148]:
print(len(labeled_examples))
print(len(unlabeled_examples))
print(len(test_examples))

35514
35513
3000


In [149]:
# Load BERT and tokenizer
transformer = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [150]:
# Main dataset function for creating semi-supervised dataset
def generate_data_loader(input_examples, label_masks, label_map, do_shuffle = False, balance_label_examples = False):

    examples = []

      # Count the percentage of labeled examples
    num_labeled_examples = 0
    for label_mask in label_masks:
        if label_mask:
            num_labeled_examples += 1
    label_mask_rate = num_labeled_examples/len(input_examples)

      # if required it applies the balance
    for index, ex in enumerate(input_examples):
        if label_mask_rate == 1 or not balance_label_examples:
            examples.append((ex, label_masks[index]))
        else:
          # IT SIMULATE A LABELED EXAMPLE
          if label_masks[index]:
            balance = int(1/label_mask_rate)
            balance = int(math.log(balance,2))
            if balance < 1:
                balance = 1
            for b in range(0, int(balance)):
                examples.append((ex, label_masks[index]))
            else:
                examples.append((ex, label_masks[index]))


    input_ids = []
    input_mask_array = []
    label_mask_array = []
    label_id_array = []

      # Tokenization
    for (text, label_mask) in tqdm.tqdm(examples):
        encoded_sent = tokenizer.encode(
            text[0], add_special_tokens=True, max_length=max_seq_length,
            padding="max_length", truncation=True)

        input_ids.append(encoded_sent)
        label_id_array.append(label_map[text[1]])
        label_mask_array.append(label_mask)

      # Attention to token
    for sent in input_ids:
        att_mask = [int(token_id > 0) for token_id in sent]
        input_mask_array.append(att_mask)

      # Convertion to Tensor
    input_ids = torch.tensor(input_ids)
    input_mask_array = torch.tensor(input_mask_array)
    label_id_array = torch.tensor(label_id_array, dtype=torch.long)
    label_mask_array = torch.tensor(label_mask_array)

      # Make the TensorDataset
    dataset = TensorDataset(
          input_ids, input_mask_array, label_id_array, label_mask_array)

    if do_shuffle:
        sampler = RandomSampler
    else:
        sampler = SequentialSampler

      # Make the DataLoader
    return DataLoader(
          dataset, sampler = sampler(dataset), batch_size = batch_size)

def format_time(elapsed):

    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))

    # Return time format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [151]:
# Define a label map for creating datasets
label_map = {}
for (i, label) in enumerate(label_list):
    label_map[label] = i


# Load trainset
train_examples = labeled_examples

# The labeled (train) dataset is assigned with a mask set to True
train_label_masks = np.ones(len(labeled_examples), dtype=bool)

# If unlabel examples are available
if unlabeled_examples:
     train_examples = train_examples + unlabeled_examples

   # The unlabeled (train) dataset is assigned with a mask set to False
     tmp_masks = np.zeros(len(unlabeled_examples), dtype=bool)
     train_label_masks = np.concatenate([train_label_masks,tmp_masks])

# Create trainloader
train_dataloader = generate_data_loader(train_examples, train_label_masks, label_map, do_shuffle = True, balance_label_examples = False)


# Load the test dataset
#The labeled (test) dataset is assigned with a mask set to True
test_label_masks = np.ones(len(test_examples), dtype=bool)

# Create test loader
test_dataloader = generate_data_loader(test_examples, test_label_masks, label_map, do_shuffle = False, balance_label_examples = False)

100%|██████████| 71027/71027 [02:10<00:00, 542.73it/s] 
/tmp/ipykernel_34/2208414523.py:54: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  label_mask_array = torch.tensor(label_mask_array)
100%|██████████| 3000/3000 [00:03<00:00, 773.33it/s] 


In [152]:
# Dataloader size
print(f'Number of training batchs with size of {batch_size}:',len(train_dataloader))

Number of training batchs with size of 128: 555


In [153]:
# # Generator architecture
# class Generator(nn.Module):
#     def __init__(self, noise_size=100, output_size=768, hidden_sizes=[512], dropout_rate=0.1):
#         super(Generator, self).__init__()
#         layers = []
#         hidden_sizes = [noise_size] + hidden_sizes

#         for i in range(len(hidden_sizes)-1):
#             layers.extend(
#                 [nn.Linear(hidden_sizes[i], hidden_sizes[i+1]),
#                  nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout_rate)])

#         layers.append(nn.Linear(hidden_sizes[-1],output_size))
#         self.layers = nn.Sequential(*layers)
#         print(self.layers)

#     def forward(self, noise):
#         output_rep = self.layers(noise)
#         return output_rep

# # Discriminator architecture
# class Discriminator(nn.Module):
#     def __init__(self, input_size=768, hidden_sizes=[512], num_labels=6, dropout_rate=0.1):
#         super(Discriminator, self).__init__()
#         self.input_dropout = nn.Dropout(p=dropout_rate)
#         layers = []
#         hidden_sizes = [input_size] + hidden_sizes
#         for i in range(len(hidden_sizes)-1):
#             layers.extend([nn.Linear(hidden_sizes[i], hidden_sizes[i+1]), nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout_rate)])

#         self.layers = nn.Sequential(*layers) #per il flatten
#         self.logit = nn.Linear(hidden_sizes[-1],num_labels+1) # +1 for the probability of this sample being fake/real.
#         self.softmax = nn.Softmax(dim=-1)

#     def forward(self, input_rep):
#         input_rep = self.input_dropout(input_rep)
#         last_rep = self.layers(input_rep)
#         logits = self.logit(last_rep)
#         probs = self.softmax(logits)
#         return last_rep, logits, probs

In [154]:
class Generator(nn.Module):
    def __init__(self, noise_dim,hidden_layers, output_layer):
        super(Generator, self).__init__()

        self.Generator_Network=nn.Sequential(nn.Linear(noise_dim,hidden_layers[0]),
                                             nn.LeakyReLU(0.2),
                                             nn.Dropout(p=0.1),
                                             nn.Linear(hidden_layers[0],output_layer))

    def forward(self,x):
        output= self.Generator_Network(x)
        return output


class Discriminator(nn.Module):
    def __init__(self,input_dim,hidden_layers, output_layer):

        super(Discriminator, self).__init__()
        self.Discriminator_Features=nn.Sequential( nn.Dropout(p=0.1),
                                                   nn.Linear(input_dim,hidden_layers[0]),
                                                   nn.LeakyReLU(0.2),
                                                   nn.Dropout(p=0.1))


        self.last_linear = nn.Linear(hidden_layers[0],output_layer)

        # self.softmax = nn.Softmax(dim=-1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):

        features = self.Discriminator_Features(x)
        last_linear_val = self.last_linear(features)
        output = self.softmax(last_linear_val)

        return features, last_linear_val, output

In [155]:
# Set config
# config = AutoConfig.from_pretrained(model_name)
# hidden_size = int(config.hidden_size)

# Define the number and width of hidden layers
# hidden_levels_g = [hidden_size for i in range(0, num_hidden_layers_g)]
# hidden_levels_d = [hidden_size for i in range(0, num_hidden_layers_d)]

# # Create generator
# generator = Generator(
#     noise_size = noise_size, output_size = hidden_size,
#     hidden_sizes = hidden_levels_g, dropout_rate = out_dropout_rate)

# # Create discriminator
# discriminator = Discriminator(
#     input_size = hidden_size, hidden_sizes = hidden_levels_d,
#     num_labels = len(label_list) , dropout_rate=out_dropout_rate)

# # Put everything in the GPU if available
# if torch.cuda.is_available():
#     generator.cuda()
#     discriminator.cuda()
#     transformer.cuda()
#     if multi_gpu:
#         transformer = torch.nn.DataParallel(transformer)

# print(config)





########################################### Navid
hidden_size=512
hidden_layers_generator=[hidden_size,hidden_size//2,hidden_size//2,hidden_size]
hidden_levels_discriminator=[hidden_size,hidden_size//2,hidden_size//2,hidden_size]


noise_dim=100
output_layer=768
generator = Generator(noise_dim=noise_dim, hidden_layers=hidden_layers_generator, output_layer=output_layer)

discriminator = Discriminator(input_dim=output_layer, hidden_layers=hidden_levels_discriminator,output_layer=len(label_list)+1)

# Put everything in the GPU if available
if torch.cuda.is_available():
    generator.cuda()
    discriminator.cuda()
    transformer.cuda()
    if multi_gpu:
        transformer = torch.nn.DataParallel(transformer)

In [156]:
print(generator.parameters)
print('----------------')
print(discriminator.parameters)

<bound method Module.parameters of Generator(
  (Generator_Network): Sequential(
    (0): Linear(in_features=100, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=768, bias=True)
  )
)>
----------------
<bound method Module.parameters of Discriminator(
  (Discriminator_Features): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=768, out_features=512, bias=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Dropout(p=0.1, inplace=False)
  )
  (last_linear): Linear(in_features=512, out_features=8, bias=True)
  (softmax): Softmax(dim=1)
)>


In [157]:
print(transformer.parameters)

<bound method Module.parameters of DataParallel(
  (module): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNor

In [158]:
# Set number of epochs
num_train_epochs = 3
multi_gpu = True
print_each_n_step = 200

# Save training/val/test stats
training_stats = []

# Measure the training time
total_t0 = time.time()

# Models parameters for passing to optimizer
transformer_vars = [i for i in transformer.parameters()]
d_vars = transformer_vars + [v for v in discriminator.parameters()]
g_vars = [v for v in generator.parameters()]

# Set optimizer
dis_optimizer = torch.optim.AdamW(d_vars, lr=learning_rate_discriminator)
gen_optimizer = torch.optim.AdamW(g_vars, lr=learning_rate_generator)

# Loop through epochs
for epoch_i in range(0, num_train_epochs):

    print("")
    print('===== Epoch {:} / {:} ====='.format(epoch_i + 1, num_train_epochs))
    print('Training...')

    # Measure epoch time
    t0 = time.time()

    # Reset the total loss for this epoch.
    tr_g_loss = 0
    tr_d_loss = 0

    # Set training mode
    transformer.train()
    generator.train()
    discriminator.train()

    # Loop through the batches
    for step, batch in tqdm.tqdm(enumerate(train_dataloader)):

        # Progress update every print_each_n_step batches.
        if step % print_each_n_step == 0 and not step == 0:

            # Calculate time
            elapsed = format_time(time.time() - t0)

            # Print the progress
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(
                step, len(train_dataloader), elapsed))

        # Unpack the training batch
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        b_label_mask = batch[3].to(device)
        real_batch_size = b_input_ids.shape[0]

        # Encode real data in the Transformer
        model_outputs = transformer(b_input_ids, attention_mask=b_input_mask)
        hidden_states = model_outputs[-1]

        # Create noise to feed to the generator
        noise = torch.randn(
            real_batch_size, noise_size, device=device)

        # Gnerate Fake data
        gen_rep = generator(noise)

        # Feed the output of the bert and the generator to disciminator
        disciminator_input = torch.cat([hidden_states, gen_rep], dim=0)

        # Get output of the disciminator
        features, logits, probs = discriminator(disciminator_input)

        # Separate the output of discriminatorfor the real and fake
        features_list = torch.split(features, real_batch_size)
        D_real_features = features_list[0]
        D_fake_features = features_list[1]

        logits_list = torch.split(logits, real_batch_size)
        D_real_logits = logits_list[0]
        D_fake_logits = logits_list[1]

        probs_list = torch.split(probs, real_batch_size)
        D_real_probs = probs_list[0]
        D_fake_probs = probs_list[1]

        # Generator LOSS
        g_loss_d = -1 * torch.mean(torch.log(1 - D_fake_probs[:,-1] + epsilon))

        g_feat_reg = torch.mean(torch.pow(
            torch.mean(D_real_features, dim=0)
             - torch.mean(D_fake_features, dim=0), 2))

        g_loss = g_loss_d + g_feat_reg

        # Disciminator LOSS
        logits = D_real_logits[:,0:-1]
        log_probs = F.log_softmax(logits, dim=-1)

        # The Loss for unlabeled data is masked out
        label2one_hot = torch.nn.functional.one_hot(b_labels, len(label_list))

        per_example_loss = -torch.sum(label2one_hot * log_probs, dim=-1)

        per_example_loss = torch.masked_select(
            per_example_loss, b_label_mask.to(device))

        labeled_example_count = per_example_loss.type(torch.float32).numel()

        if labeled_example_count == 0:
            D_L_Supervised = 0

        else:
            D_L_Supervised = torch.div(
              torch.sum(per_example_loss.to(device)), labeled_example_count)

        D_L_unsupervised1U = -1 * torch.mean(
            torch.log(1 - D_real_probs[:, -1] + epsilon))

        D_L_unsupervised2U = -1 * torch.mean(
            torch.log(D_fake_probs[:, -1] + epsilon))

        d_loss = D_L_Supervised + D_L_unsupervised1U + D_L_unsupervised2U

        # Reset gradients
        gen_optimizer.zero_grad()
        dis_optimizer.zero_grad()

        # Backward pass
        g_loss.backward(retain_graph=True)
        d_loss.backward()

        # Update weights
        gen_optimizer.step()
        dis_optimizer.step()

        # Save the losses to report
        tr_g_loss += g_loss.item()
        tr_d_loss += d_loss.item()


    # Calculate the average loss over the batches.
    avg_train_loss_g = tr_g_loss / len(train_dataloader)
    avg_train_loss_d = tr_d_loss / len(train_dataloader)

    # Measure epoch time
    training_time = format_time(time.time() - t0)

    # Pritn stats
    print("")
    print("Average training loss generetor: {0:.3f}".format(avg_train_loss_g))
    print("Average training loss discriminator: {0:.3f}".format(avg_train_loss_d))
    print("Training epcoh took: {:}".format(training_time))


    # Test
    print("")
    print("Running Test...")

    t0 = time.time()

    # Set validation mode
    transformer.eval()
    discriminator.eval()
    generator.eval()

    # Tracking variables
    total_test_accuracy = 0

    total_test_loss = 0
    nb_test_steps = 0

    all_preds = []
    all_labels_ids = []

    # Define Loss function
    nll_loss = torch.nn.CrossEntropyLoss(ignore_index=-1)

    # Loop through test patches
    for batch in test_dataloader:

        # Unpack the test batch
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # No grad block
        with torch.no_grad():
            model_outputs = transformer(
                b_input_ids, attention_mask=b_input_mask)

            hidden_states = model_outputs[-1]
            _, logits, probs = discriminator(hidden_states)

            filtered_logits = logits[:,0:-1]

            # Accumulate the test loss.
            total_test_loss += nll_loss(filtered_logits, b_labels)

        # Accumulate the predictions and the input labels
        _, preds = torch.max(filtered_logits, 1)
        all_preds += preds.detach().cpu()
        all_labels_ids += b_labels.detach().cpu()

    # Report the final accuracy for this validation run.
    all_preds = torch.stack(all_preds).numpy()
    all_labels_ids = torch.stack(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print(" Test Accuracy: {0:.3f}".format(test_accuracy))

    # Calculate the average loss over all of the batches.
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()

    # Measure validation runtime
    test_time = format_time(time.time() - t0)

    # Print validation stats
    print("  Test Loss: {0:.3f}".format(avg_test_loss))
    print("  Test took: {:}".format(test_time))

    # Store stats from this epoch.
    training_stats.append(
        {'epoch': epoch_i + 1, 'Training Loss generator': avg_train_loss_g,
         'Training Loss discriminator': avg_train_loss_d,
         'Valid. Loss': avg_test_loss, 'Valid. Accur.': test_accuracy,
         'Training Time': training_time, 'Test Time': test_time})


===== Epoch 1 / 3 =====
Training...


200it [07:27,  2.24s/it]

  Batch   200  of    555.    Elapsed: 0:07:28.


400it [14:55,  2.24s/it]

  Batch   400  of    555.    Elapsed: 0:14:56.


...

  Batch   200  of    555.    Elapsed: 0:07:27.


400it [14:54,  2.23s/it]

  Batch   400  of    555.    Elapsed: 0:14:55.


555it [20:41,  2.24s/it]



Average training loss generetor: 0.697
Average training loss discriminator: 0.897
Training epcoh took: 0:20:41

Running Test...
 Test Accuracy: 0.518
  Test Loss: 2.470
  Test took: 0:00:11


In [161]:
for stat in training_stats:
    print(stat)

print("\nTraining complete!")
print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

{'epoch': 1, 'Training Loss generator': 0.6931096321022189, 'Training Loss discriminator': 1.5931815234390465, 'Valid. Loss': 1.637075424194336, 'Valid. Accur.': 0.49433333333333335, 'Training Time': '0:20:43', 'Test Time': '0:00:11'}
{'epoch': 2, 'Training Loss generator': 0.6970585456839553, 'Training Loss discriminator': 1.075995552217638, 'Valid. Loss': 2.136740207672119, 'Valid. Accur.': 0.494, 'Training Time': '0:20:42', 'Test Time': '0:00:11'}
{'epoch': 3, 'Training Loss generator': 0.6967352819872332, 'Training Loss discriminator': 0.8969939951424126, 'Valid. Loss': 2.469900131225586, 'Valid. Accur.': 0.518, 'Training Time': '0:20:41', 'Test Time': '0:00:11'}

Training complete!
Total training took 1:03:29 (h:mm:ss)


In [164]:
# Save training and validation/test stats to report later
# and comprasion with other parts

# Write stats as a json file
with open("Part3_stats_50persent_maxlen128.json", "w") as outfile:
     json.dump(training_stats, outfile)